# Day 13

## 1. Define the required fields for each file

In [1]:
input_folder = "../data"

In [2]:
REQUIRED_FIELDS = {
    "olist_customers_dataset.csv": {"customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"},
    "olist_customers_semicolon.csv": {"customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"},
    "olist_geolocation_dataset.csv": {"geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_city", "geolocation_state"},
    "olist_order_items_dataset.csv": {"order_id", "order_item_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value"},
    "olist_order_payments_dataset.csv": {"order_id", "payment_sequential", "payment_type", "payment_installments", "payment_value"},
    "olist_order_reviews_dataset.csv": {"review_id", "order_id", "review_score", "review_creation_date", "review_answer_timestamp"},
    "olist_orders_dataset.csv": {"order_id", "customer_id", "order_status", "order_purchase_timestamp"},
    "olist_products_dataset.csv": {"product_id"},
    "olist_sellers_dataset.csv": {"seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"},
    "product_category_name_translation.csv": {"product_category_name", "product_category_name_english"},
}

## 2. Validate every row and collect errors

In [3]:
import csv
import re
from datetime import datetime
from pathlib import Path

all_rows = []

messy_files = {
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_sellers_dataset.csv",
}

for file_name in messy_files:
    required_fields = REQUIRED_FIELDS[file_name]
    path = Path(input_folder) / file_name

    if not path.is_file() or path.stat().st_size == 0:
        print(f"Skipped {file_name}: file is missing or empty")
        continue

    with path.open("r", newline="", encoding="utf-8-sig") as file:
        first_line = file.readline()
        file.seek(0)
        delimiter = ";" if ";" in first_line else ","
        reader = csv.DictReader(file, delimiter=delimiter)

        for row_number, row in enumerate(reader, start=2):
            record = {
                "file": file_name,
                "row": row_number,
                "data": row,
                "reasons": [],
            }

            for field in required_fields:
                if not row.get(field):
                    record["reasons"].append(f"missing {field}")

            all_rows.append(record)

print(f"Read {len(all_rows)} rows")

Read 100 rows


## 4. Check data types

In [4]:
for record in all_rows:
    for field, value in record["data"].items():
        if "date" in field or "timestamp" in field or field.endswith("_at"):
            if value:
                try:
                    datetime.fromisoformat(value)
                except ValueError:
                    record["reasons"].append(f"invalid date in {field}")

    if "price" in record["data"]:
        try:
            float(record["data"]["price"])
        except ValueError:
            record["reasons"].append("price is not numeric")

## 5. Apply business rules

In [5]:
allowed_status = {"created", "approved", "invoiced", "processing", "shipped", "delivered", "unavailable", "canceled"}

for record in all_rows:
    row = record["data"]

    if "price" in row:
        try:
            if float(row["price"]) <= 0:
                record["reasons"].append("price must be greater than 0")
        except ValueError:
            pass

    if "order_status" in row:
        if row["order_status"] not in allowed_status:
            record["reasons"].append("invalid order_status")

## 6. Check zip prefixes and IDs with regex

In [6]:
for record in all_rows:
    for field, value in record["data"].items():
        if value:
            if "zip" in field or field.endswith("_prefix"):
                if re.fullmatch(r"\d{5}", value) is None:
                    record["reasons"].append(f"invalid zip prefix in {field}")

            if (field.endswith("_id") and field != "order_item_id") or field == "review_id":
                if re.fullmatch(r"[0-9a-fA-F]{32}", value) is None:
                    record["reasons"].append(f"invalid ID in {field}")

bad_rows = []
for record in all_rows:
    if record["reasons"]:
        bad_rows.append({
            "file": record["file"],
            "row": record["row"],
            "reason": "; ".join(record["reasons"]),
            "data": record["data"],
        })

print(f"Found {len(bad_rows)} rejected rows")

Found 32 rejected rows


## 3. Write rejected rows to bad_records.csv with a reason

In [10]:
with open("../bad_records.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["file", "row", "reason", "data"])

    for bad_row in bad_rows:
        writer.writerow([bad_row["file"], bad_row["row"], bad_row["reason"], bad_row["data"]])

print(f"Wrote {len(bad_rows)} rejected rows")

Wrote 32 rejected rows


## 7. Create custom exceptions

In [8]:
class SchemaError(Exception):
    pass

class FileFormatError(Exception):
    pass

## 8. Handle missing or empty files

In [9]:
for file_name in REQUIRED_FIELDS:
    path = Path(input_folder) / file_name

    if not path.is_file():
        print(f"Skipped {file_name}: file is missing")
    elif path.stat().st_size == 0:
        print(f"Skipped {file_name}: file is empty")

## 9. Print the validation summary

In [11]:
total_rows = len(all_rows)
rejected_rows = len(bad_rows)
valid_rows = total_rows - rejected_rows

print(f"Total: {total_rows}")
print(f"Valid: {valid_rows}")
print(f"Rejected: {rejected_rows}")

Total: 100
Valid: 68
Rejected: 32


## 10. Fail the whole file if the rejected percentage is over 20%

In [12]:
rejection_limit = 0.20

for file_name in messy_files:
    file_rows = [row for row in all_rows if row["file"] == file_name]
    rejected = [row for row in file_rows if row["reasons"]]

    if file_rows:
        rejected_percent = len(rejected) / len(file_rows)
        if rejected_percent > rejection_limit:
            print(f"FAIL: {file_name} has more than {rejection_limit * 100:.0f}% rejected rows")
        else:
            print(f"PASS: {file_name}")

FAIL: olist_orders_dataset.csv has more than 20% rejected rows
FAIL: olist_customers_dataset.csv has more than 20% rejected rows
FAIL: olist_order_items_dataset.csv has more than 20% rejected rows
FAIL: olist_sellers_dataset.csv has more than 20% rejected rows
PASS: olist_geolocation_dataset.csv
